In [1]:
from dataset import *
from gpr import *
from train import *

In [3]:
dataset = ACE_Dataset(
    target_y="TOTEN",
    directory="",
    max_order=3,
)

100%|██████████| 250/250 [02:12<00:00,  1.88it/s]


In [4]:
train_dataset, valid_dataset = train_valid_split(dataset,
                                                 train_fraction=0.8,
                                                 seed=24153
                                                 )

train_x, train_y = get_tensors_from_subset(train_dataset)
valid_x, valid_y = get_tensors_from_subset(valid_dataset)

In [5]:
torch.set_default_dtype(torch.float64)

device = torch.device("cpu")
dtype = torch.float64

atom_dim = train_x[0].shape[1]

likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device=device, dtype=dtype)

model = AtomicSumExactGP(
    n_train=len(train_x),
    train_y=train_y,
    likelihood=likelihood,
    atom_dim=atom_dim,
).to(device=device, dtype=dtype)

model.covar_module.set_train_structures(
    train_x,
    device=device,
    dtype=dtype,
)

model.covar_module.atom_kernel.base_kernel.lengthscale = torch.ones(
    atom_dim,
    device=device,
    dtype=dtype,
)

model.covar_module.atom_kernel.outputscale = torch.tensor(
    1.0,
    device=device,
    dtype=dtype,
)

likelihood.noise = torch.tensor(
    1e-2,
    device=device,
    dtype=dtype,
)

n_atoms_train = torch.tensor(
    [x.shape[0] for x in train_x],
    dtype=dtype,
    device=device,
)

model.mean_module.constant.data = (train_y / n_atoms_train).mean()

In [6]:
n_epoch = 100 # <- hyperparameter

In [33]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=100
)

In [40]:
logs, best_mae_valid = train(
    train_x=train_x,
    train_y=train_y,
    valid_x=valid_x,
    valid_y=valid_y,
    optimizer=optimizer,
    scheduler=scheduler,
    model=model,
    n_epochs = n_epoch,
    device = device,
    metadata = {"system": "Pt-Pd slab",
                "dataset_size": len(train_x),
                "target_y": "TOTEN",
                },
)

Iter 1/100 Loss: -0.435334 MAE train: 0.000760 MAE valid: 0.049346 best MAE valid: 0.049346 noise: 0.000100 lr: 5.000e-01
Iter 2/100 Loss: -0.447675 MAE train: 0.000761 MAE valid: 0.048718 best MAE valid: 0.048718 noise: 0.000100 lr: 5.000e-01
Iter 3/100 Loss: -0.459734 MAE train: 0.000764 MAE valid: 0.048107 best MAE valid: 0.048107 noise: 0.000100 lr: 5.000e-01
Iter 4/100 Loss: -0.471599 MAE train: 0.000769 MAE valid: 0.047512 best MAE valid: 0.047512 noise: 0.000100 lr: 5.000e-01
Iter 5/100 Loss: -0.483344 MAE train: 0.000779 MAE valid: 0.046934 best MAE valid: 0.046934 noise: 0.000100 lr: 5.000e-01
Iter 6/100 Loss: -0.495023 MAE train: 0.000789 MAE valid: 0.046373 best MAE valid: 0.046373 noise: 0.000100 lr: 5.000e-01
Iter 7/100 Loss: -0.506642 MAE train: 0.000802 MAE valid: 0.045826 best MAE valid: 0.045826 noise: 0.000100 lr: 5.000e-01
Iter 8/100 Loss: -0.518177 MAE train: 0.000817 MAE valid: 0.045291 best MAE valid: 0.045291 noise: 0.000100 lr: 5.000e-01
Iter 9/100 Loss: -0.5295

In [7]:
checkpoint = torch.load("best_gpr_model_slab.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
model.likelihood.load_state_dict(checkpoint["likelihood_state_dict"])

<All keys matched successfully>

# Supercell test

In [31]:
from ase.build import make_supercell

matrix = [[3,0,0],[0,3,0],[0,0,1]]

supercell = [torch.Tensor(Cluster_Expansion(make_supercell(atoms[i], matrix),
                            shells=shells,
                            max_order=3,
                            atom_indices=None
                           ).descriptor) for i in tqdm(range(len(atoms)))]

100%|██████████| 20/20 [01:31<00:00,  4.59s/it]


In [32]:
model.eval()
likelihood.eval()

with model.covar_module.register_query_structures(supercell) as test_ids:
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred = likelihood(model(test_ids))

mean = pred.mean
std = pred.stddev

print("mean =", mean)
print("std  =", std)

mean = tensor([-1403.1064, -1525.4287, -1333.9101, -1410.1130, -1317.3994, -1734.3997,
        -1192.0357, -1650.5030, -1412.2342, -1526.5610, -1715.8978, -1434.6880,
        -1426.0491, -1199.9771, -1863.1740, -1794.5319, -1881.0557, -1338.9011,
        -1734.4760, -1574.7308])
std  = tensor([0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350,
        0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350, 0.1350,
        0.1350, 0.1350])


In [33]:
y*9

tensor([-1403.1065, -1525.4287, -1333.9098, -1410.1130, -1317.3997, -1734.3998,
        -1192.0344, -1650.5032, -1412.2341, -1526.5611, -1715.8975, -1434.6881,
        -1426.0490, -1199.9780, -1863.1733, -1794.5322, -1881.0564, -1338.9008,
        -1734.4762, -1574.7307])

# Validation

In [14]:
from ase.io import read
from calculator import *

calc = Calculator(model = model,
                  mode = "TOTEN"
                  )

atoms = read("/Users/oq7884/Documents/for_Danila/seminar/Random_9layers/alloy_233/OUTCAR")

print(f'Predicted energy = {calc(atoms)[0][0]:.4f} eV')
print(f'DFT energy = {atoms.get_potential_energy():.4f} eV')

print(f'MAE = {np.abs(calc(atoms)[0][0] - atoms.get_potential_energy())*1000:.2f} meV')

Predicted energy = -151.1200 eV
DFT energy = -151.1247 eV
MAE = 4.75 meV
